# AC40 Regatta Replay — Interactive Study Notebook (Pre-Executed)
**Data Provenance**: Combines **actual repository evidence** (`evidence/naples_pr1_replay_bounded_excerpt.json`, `evidence/naples_pr1_performance.csv`, `evidence/naples_mr_final_performance.csv`, `evidence/scheduled_entry_ledger.json`, `evidence/cup_2027.json`) with **synthetic edge-case tests**.

All cells below have been executed and verified against the frozen `2026-11-04` snapshot.

In [1]:
#!/usr/bin/env python3
"""
Executable Example 01 — WGS84 Curvature Projection, Circular Interpolation & Bow Crossing
=========================================================================================
Data Provenance:
  - PART A uses ACTUAL REPOSITORY EVIDENCE from `evidence/naples_pr1_replay_bounded_excerpt.json`
    (Naples course origin, WGS84 scale factors `k_lat` and `k_lon`, and FRA start crossing).
  - PART B uses SYNTHETIC DATA to demonstrate shortest-arc circular interpolation across 0/360 deg
    and exact sub-frame bow-tip (+12.963 m) segment intersection math.

Requires: Python 3.10+ standard library only (no third-party packages).
"""

from __future__ import annotations

import json
import math
from pathlib import Path

EVIDENCE_DIR = Path(__file__).resolve().parent.parent / "evidence"

# WGS84 constants matching pipeline/projection.py:L14-L16
WGS84_A = 6_378_137.0
WGS84_F = 1.0 / 298.257223563
WGS84_E2 = 2.0 * WGS84_F - WGS84_F * WGS84_F
BOW_OFFSET_M = 12.963  # AC40 transom-to-bowsprit tip (pipeline/reconstruction.py:L25)


def wgs84_scale_factors(lat0_deg: float) -> tuple[float, float]:
    """Return (k_lon, k_lat) in metres per degree at reference latitude lat0_deg."""
    phi = math.radians(lat0_deg)
    sin_phi = math.sin(phi)
    denom = 1.0 - WGS84_E2 * sin_phi * sin_phi
    m_radius = WGS84_A * (1.0 - WGS84_E2) / (denom ** 1.5)
    n_radius = WGS84_A / math.sqrt(denom)
    k_lon = math.radians(1.0) * n_radius * math.cos(phi)
    k_lat = math.radians(1.0) * m_radius
    return k_lon, k_lat


def interpolate_angle_deg(a0: float, a1: float, alpha: float) -> float:
    """Shortest-arc circular interpolation in [0, 360) (pipeline/synchronize.py:L29-L48)."""
    delta = ((a1 - a0 + 540.0) % 360.0) - 180.0
    return (a0 + alpha * delta) % 360.0


def bow_point(x: float, y: float, heading_deg: float, offset_m: float = BOW_OFFSET_M) -> tuple[float, float]:
    """Project transom GNSS (x, y) forward along true heading to bowsprit tip."""
    rad = math.radians(heading_deg)
    return (x + offset_m * math.sin(rad), y + offset_m * math.cos(rad))


def segment_intersection_fraction(
    p0: tuple[float, float],
    p1: tuple[float, float],
    q0: tuple[float, float],
    q1: tuple[float, float],
) -> tuple[float, float] | None:
    """
    Exact 2D segment intersection matching pipeline/reconstruction.py:L39-L73.
    Returns (u, v) in [0, 1]^2 where u is fraction along motion p0->p1
    and v is fraction along gate segment q0->q1, or None if non-intersecting.
    """
    rx, ry = p1[0] - p0[0], p1[1] - p0[1]
    sx, sy = q1[0] - q0[0], q1[1] - q0[1]
    rxs = rx * sy - ry * sx
    if abs(rxs) < 1e-12:
        return None
    qpx, qpy = q0[0] - p0[0], q0[1] - p0[1]
    u = (qpx * sy - qpy * sx) / rxs
    v = (qpx * ry - qpy * rx) / rxs
    if -1e-9 <= u <= 1.0 + 1e-9 and -1e-9 <= v <= 1.0 + 1e-9:
        return max(0.0, min(1.0, u)), max(0.0, min(1.0, v))
    return None


def main() -> None:
    print("=" * 78)
    print("EXECUTABLE EXAMPLE 01: PROJECTION, CIRCULAR INTERPOLATION & BOW CROSSING")
    print("=" * 78)

    # PART A: Actual Repository Evidence (naples_pr1)
    excerpt_path = EVIDENCE_DIR / "naples_pr1_replay_bounded_excerpt.json"
    excerpt = json.loads(excerpt_path.read_text(encoding="utf-8"))
    origin = excerpt["course"]["origin"]
    proj = excerpt["course"]["projection"]
    k_lon, k_lat = wgs84_scale_factors(origin["lat"])

    print("\n[PART A — ACTUAL REPOSITORY EVIDENCE: naples_pr1]")
    print(f"  Session Course Origin    : ({origin['lat']:.8f} N, {origin['lon']:.8f} E)")
    print(f"  WGS84 Analytical Curv.   : k_lon = {k_lon:.3f} m/deg, k_lat = {k_lat:.3f} m/deg")
    print(f"  Exported course.projection: k_lon = {proj['k_lon']:.3f} m/deg, k_lat = {proj['k_lat']:.3f} m/deg")
    assert abs(k_lon - proj["k_lon"]) < 5.0
    assert abs(k_lat - proj["k_lat"]) < 5.0

    cag_lon, cag_lat = wgs84_scale_factors(39.1900)
    print(f"  CAGLIARI_REFERENCE(39.19N): k_lon = {cag_lon:.3f} m/deg, k_lat = {cag_lat:.3f} m/deg")

    gun_ms = excerpt["chronology"]["gun_time"]
    fra = next(c for c in excerpt["competitors"] if c["team_code"] == "FRA")
    fra_cross_ms = fra["start"]["crossing_time"]
    print(f"  naples_pr1 Gun Time      : {gun_ms} ms")
    print(f"  FRA Start Crossing Time  : {fra_cross_ms} ms ( Gun + {(fra_cross_ms - gun_ms)/1000:.3f} s )")
    assert fra_cross_ms - gun_ms == 436

    # PART B: Synthetic Hand-Checkable Examples
    print("\n[PART B — SYNTHETIC HAND-CHECKABLE EXAMPLES]")
    # 1. Shortest-arc circular interpolation across 0 deg North
    h0, h1, alpha = 355.0, 5.0, 0.30
    naive_linear = h0 + alpha * (h1 - h0)
    circular = interpolate_angle_deg(h0, h1, alpha)
    print(f"  1. Heading wrap 355 deg -> 5 deg at alpha=0.30:")
    print(f"     Naive linear lerp (WRONG) : {naive_linear:6.2f} deg (points South!)")
    print(f"     Shortest-arc angleLerp    : {circular:6.2f} deg (correct: 358.00 deg)")
    assert abs(circular - 358.0) < 1e-9

    # 2. Sub-frame bow crossing vs transom GNSS crossing
    q0, q1 = (-50.0, 100.0), (50.0, 100.0)
    t0, t1 = 1000, 1100
    transom0, transom1 = (0.0, 86.5), (0.0, 87.5)
    bow0 = bow_point(*transom0, heading_deg=0.0)
    bow1 = bow_point(*transom1, heading_deg=0.0)
    transom_hit = segment_intersection_fraction(transom0, transom1, q0, q1)
    bow_hit = segment_intersection_fraction(bow0, bow1, q0, q1)
    assert transom_hit is None and bow_hit is not None
    u, v = bow_hit
    t_cross = round(t0 + u * (t1 - t0))
    print(f"  2. Sub-frame bow crossing across gate y=100 m between t0=1000 ms and t1=1100 ms:")
    print(f"     Transom y0 -> y1          : {transom0[1]:.3f} m -> {transom1[1]:.3f} m (hit={transom_hit})")
    print(f"     Bowsprit y0 -> y1 (+12.963): {bow0[1]:.3f} m -> {bow1[1]:.3f} m")
    print(f"     Intersection fractions    : u = {u:.4f}, v = {v:.4f} -> t_cross = {t_cross} ms")
    assert t_cross == 1054
    print("\nAll assertions passed successfully.")


if __name__ == "__main__":
    main()


EXECUTABLE EXAMPLE 01: PROJECTION, CIRCULAR INTERPOLATION & BOW CROSSING

[PART A — ACTUAL REPOSITORY EVIDENCE: naples_pr1]
  Session Course Origin    : (40.80948067 N, 14.23726898 E)
  WGS84 Analytical Curv.   : k_lon = 84376.984 m/deg, k_lat = 111050.227 m/deg
  Exported course.projection: k_lon = 84374.839 m/deg, k_lat = 111047.746 m/deg
  CAGLIARI_REFERENCE(39.19N): k_lon = 86394.247 m/deg, k_lat = 111019.109 m/deg
  naples_pr1 Gun Time      : 1790079000000 ms
  FRA Start Crossing Time  : 1790079000436 ms ( Gun + 0.436 s )

[PART B — SYNTHETIC HAND-CHECKABLE EXAMPLES]
  1. Heading wrap 355 deg -> 5 deg at alpha=0.30:
     Naive linear lerp (WRONG) : 250.00 deg (points South!)
     Shortest-arc angleLerp    : 358.00 deg (correct: 358.00 deg)
  2. Sub-frame bow crossing across gate y=100 m between t0=1000 ms and t1=1100 ms:
     Transom y0 -> y1          : 86.500 m -> 87.500 m (hit=None)
     Bowsprit y0 -> y1 (+12.963): 99.463 m -> 100.463 m
     Intersection fractions    : u = 0.53

In [2]:
#!/usr/bin/env python3
"""
Executable Example 02 — Reconciling the 6-Axis Peer-Normalized Performance Radar
================================================================================
Data Provenance:
  - Uses ACTUAL REPOSITORY EVIDENCE from:
      1. `evidence/naples_pr1_performance.csv`
      2. `evidence/naples_mr_final_performance.csv`
  - Demonstrates:
      a) Self-excluding peer mean on each matched leg.
      b) Why the race dimension index (mean of per-leg ratios) differs from 100 * value / peer_mean
         (ratio of cross-leg means) using `ITA G` Downwind VMG in `naples_pr1` (`109.9671` vs `108.0853`).
      c) Why Leg 2 has `peer_count = 6` while Leg 4 has `peer_count = 4` (`NZL B` and `SUI` drop out
         of Leg 4 due to `< 95%` valid source coverage).
      d) Why `naples_mr_final` (`ITA B` vs `NZL B`) restricts scope to 4 of 6 legs (`matched_legs = [1,2,3,4]`).

Requires: Python 3.10+ standard library only.
"""

from __future__ import annotations

import csv
from pathlib import Path

EVIDENCE_DIR = Path(__file__).resolve().parent.parent / "evidence"


def load_csv(name: str) -> list[dict[str, str]]:
    with (EVIDENCE_DIR / name).open("r", encoding="utf-8", newline="") as f:
        return list(csv.DictReader(f))


def main() -> None:
    print("=" * 78)
    print("EXECUTABLE EXAMPLE 02: SELF-EXCLUDING PEER RADAR & MATCHED-LEG SCOPE")
    print("=" * 78)

    pr1_rows = load_csv("naples_pr1_performance.csv")

    # 1. Extract ITA G downwind_vmg rows (all_comparable_legs, leg 2, leg 4)
    ita_all = next(
        r for r in pr1_rows
        if r["team_code"] == "ITA G" and r["scope"] == "all_comparable_legs" and r["metric"] == "downwind_vmg"
    )
    ita_l2 = next(
        r for r in pr1_rows
        if r["team_code"] == "ITA G" and r["scope"] == "leg" and r["leg"] == "2" and r["metric"] == "downwind_vmg"
    )
    ita_l4 = next(
        r for r in pr1_rows
        if r["team_code"] == "ITA G" and r["scope"] == "leg" and r["leg"] == "4" and r["metric"] == "downwind_vmg"
    )

    val_l2, peer_l2, idx_l2 = float(ita_l2["value"]), float(ita_l2["peer_mean"]), float(ita_l2["index"])
    val_l4, peer_l4, idx_l4 = float(ita_l4["value"]), float(ita_l4["peer_mean"]), float(ita_l4["index"])
    val_all, peer_all, idx_all = float(ita_all["value"]), float(ita_all["peer_mean"]), float(ita_all["index"])

    mean_of_ratios = (idx_l2 + idx_l4) / 2.0
    ratio_of_means = 100.0 * val_all / peer_all

    print("\n[1. ACTUAL REPOSITORY EVIDENCE: naples_pr1 — ITA G Downwind VMG]")
    print(f"  Leg 2 (peer_count={ita_l2['peer_count']}): ITA G = {val_l2:.4f} kt | Peer Mean = {peer_l2:.4f} kt | Leg 2 Index = {idx_l2:.4f}")
    print(f"  Leg 4 (peer_count={ita_l4['peer_count']}): ITA G = {val_l4:.4f} kt | Peer Mean = {peer_l4:.4f} kt | Leg 4 Index = {idx_l4:.4f}")
    print(f"  Exported Dimension Value (Mean of Legs 2 & 4)     : {val_all:.4f} kt")
    print(f"  Exported Dimension Peer Mean (Mean of Peer Means) : {peer_all:.4f} kt")
    print(f"  Exported Dimension Index (Mean of Leg Ratios)     : {idx_all:.4f} (computed: {mean_of_ratios:.4f})")
    print(f"  Naive 100 * value / peer_mean (Ratio of Means)    : {ratio_of_means:.4f}")
    print(f"  Discrepancy (Mean-of-Ratios minus Ratio-of-Means) : +{idx_all - ratio_of_means:.4f} index pts")

    assert abs(idx_all - 109.9671042123199) < 1e-9
    assert abs(ratio_of_means - 108.085343) < 1e-5

    # 2. Verify USA and GBR B exclusion reasons in naples_pr1
    usa_row = next(r for r in pr1_rows if r["team_code"] == "USA" and r["scope"] == "all_comparable_legs")
    gbr_b_row = next(r for r in pr1_rows if r["team_code"] == "GBR B" and r["scope"] == "all_comparable_legs")
    print(f"\n  Excluded boats in naples_pr1:")
    print(f"    USA   quality={usa_row['quality']} | reason='{usa_row['reason']}'")
    print(f"    GBR B quality={gbr_b_row['quality']} | reason='{gbr_b_row['reason']}'")

    # 3. Verify Match Race Final (naples_mr_final) matched-leg scope
    mr_rows = load_csv("naples_mr_final_performance.csv")
    print("\n[2. ACTUAL REPOSITORY EVIDENCE: naples_mr_final (ITA B vs NZL B)]")
    for r in mr_rows:
        if r["team_code"] == "NZL B" and r["scope"] == "all_comparable_legs":
            print(
                f"  NZL B | {r['metric']:16s} | index={float(r['index']):8.2f} | "
                f"val={float(r['value']):6.2f} {r['unit']:2s} | peer={float(r['peer_mean']):6.2f} | "
                f"legs_count={r['sample_count']} | coverage_s={float(r['coverage_s']):6.2f}s"
            )
    print("\nAll assertions passed successfully.")


if __name__ == "__main__":
    main()


EXECUTABLE EXAMPLE 02: SELF-EXCLUDING PEER RADAR & MATCHED-LEG SCOPE

[1. ACTUAL REPOSITORY EVIDENCE: naples_pr1 — ITA G Downwind VMG]
  Leg 2 (peer_count=6): ITA G = 19.5725 kt | Peer Mean = 15.4085 kt | Leg 2 Index = 127.0241
  Leg 4 (peer_count=4): ITA G = 17.8665 kt | Peer Mean = 19.2299 kt | Leg 4 Index = 92.9101
  Exported Dimension Value (Mean of Legs 2 & 4)     : 18.7195 kt
  Exported Dimension Peer Mean (Mean of Peer Means) : 17.3192 kt
  Exported Dimension Index (Mean of Leg Ratios)     : 109.9671 (computed: 109.9671)
  Naive 100 * value / peer_mean (Ratio of Means)    : 108.0853
  Discrepancy (Mean-of-Ratios minus Ratio-of-Means) : +1.8818 index pts

  Excluded boats in naples_pr1:
    USA   quality=unavailable | reason='boat has no eligible racing window or has a source-state conflict'
    GBR B quality=unavailable | reason='boat has no eligible racing window or has a source-state conflict'

[2. ACTUAL REPOSITORY EVIDENCE: naples_mr_final (ITA B vs NZL B)]
  NZL B | upwind_

In [3]:
#!/usr/bin/env python3
"""
Executable Example 03 — Plackett-Luce Expected Rank, Tied-N Identity & AC38 Cup Bounds
======================================================================================
Data Provenance:
  - PART A uses SYNTHETIC 3-BOAT & 2-BOAT EXAMPLES to prove:
      1. The Tied-N Expected Rank Sum Identity vs Standard Permutation Rank Sum K(K+1)/2.
      2. Persistent vs Independent Ability Draws in a First-to-7 Cup Match (75.00% vs 93.76%).
  - PART B uses ACTUAL REPOSITORY EVIDENCE from `evidence/cup_2027.json` and
    `evidence/scheduled_entry_ledger.json` to verify the 227-entry ledger and the
    34.46% RRSAC 102.1 unresolved interval width across all 7 AC38 teams.

Requires: Python 3.10+ standard library only.
"""

from __future__ import annotations

import json
import math
from pathlib import Path

EVIDENCE_DIR = Path(__file__).resolve().parent.parent / "evidence"


def analytic_pl_expected_ranks_with_tied_n(
    gammas: list[float], p_complete: list[float]
) -> tuple[list[float], float]:
    """
    Exact enumeration over all 2^K completion subsets S, using the analytic Plackett-Luce
    conditional expected rank formula E[R_i | S] = 1 + sum_{j in S, j != i} gamma_j / (gamma_i + gamma_j)
    for finishers i in S, and rank K for non-finishers i not in S (pipeline/prediction/simulation.py).
    """
    k = len(gammas)
    expected_ranks = [0.0] * k
    excess_sum_from_formula = 0.0

    for mask in range(1 << k):
        prob_s = 1.0
        finishers = []
        for i in range(k):
            if (mask >> i) & 1:
                prob_s *= p_complete[i]
                finishers.append(i)
            else:
                prob_s *= 1.0 - p_complete[i]

        m = len(finishers)
        if m <= k - 2:
            excess_sum_from_formula += prob_s * ((k - m) * (k - m - 1) / 2.0)

        for i in range(k):
            if (mask >> i) & 1:
                cond_rank = 1.0 + sum(
                    gammas[j] / (gammas[i] + gammas[j]) for j in finishers if j != i
                )
            else:
                cond_rank = float(k)  # Tied-N nonfinish convention
            expected_ranks[i] += prob_s * cond_rank

    return expected_ranks, excess_sum_from_formula


def first_to_7_win_prob_given_p(p: float) -> float:
    """Negative binomial probability that Team A reaches 7 wins before Team B reaches 7 wins."""
    total = 0.0
    for b_wins in range(7):
        total += math.comb(6 + b_wins, b_wins) * (p ** 7) * ((1.0 - p) ** b_wins)
    return total


def main() -> None:
    print("=" * 78)
    print("EXECUTABLE EXAMPLE 03: PLACKETT-LUCE, TIED-N SUM & AC38 CUP RULE BOUNDS")
    print("=" * 78)

    # PART A.1: Synthetic 3-boat Tied-N Rank Sum Identity
    gammas = [3.0, 2.0, 1.0]
    p_comp = [0.90, 0.85, 0.75]
    k = len(gammas)
    exp_ranks, excess = analytic_pl_expected_ranks_with_tied_n(gammas, p_comp)
    base_sum = k * (k + 1) / 2.0

    print("\n[PART A.1 — SYNTHETIC 3-BOAT TIED-N EXPECTED RANK SUM IDENTITY]")
    for i, r in enumerate(exp_ranks, 1):
        print(f"  Boat {i} (gamma={gammas[i-1]:.1f}, p_comp={p_comp[i-1]:.2f}) -> E[R_{i}] = {r:.6f}")
    print(f"  Standard permutation sum K(K+1)/2 : {base_sum:.6f}")
    print(f"  Theoretical tied-N excess         : +{excess:.6f}")
    print(f"  Sum of expected ranks             : {sum(exp_ranks):.6f}")
    assert abs(sum(exp_ranks) - (base_sum + excess)) < 1e-12

    # PART A.2: Persistent vs Independent Ability Draws in a First-to-7 Series
    # (docs/forecasting_implementation.md:L122-L123)
    # Under a 50/50 mixture of latent single-race win probability p in {0.50, 0.90}:
    # - Persistent tournament draw: 0.5 * P(series | p=0.50) + 0.5 * P(series | p=0.90) = 75.00%
    # - Independent per-race draws: marginal p_bar = 0.5*(0.50 + 0.90) = 0.70 -> P(series | p=0.70) = 93.76%
    series_persistent = 0.5 * first_to_7_win_prob_given_p(0.50) + 0.5 * first_to_7_win_prob_given_p(0.90)
    series_independent = first_to_7_win_prob_given_p(0.70)
    print("\n[PART A.2 — PERSISTENT VS INDEPENDENT SERIES WIN PROBABILITY (FIRST-TO-7)]")
    print(f"  50/50 mix of latent p in {{0.50, 0.90}} — Persistent ability draw : {series_persistent*100:.2f}%")
    print(f"  50/50 mix of latent p in {{0.50, 0.90}} — Independent draws (p=0.7): {series_independent*100:.2f}%")
    assert round(series_persistent * 100, 2) == 75.00
    assert round(series_independent * 100, 2) == 93.76

    # PART B: Actual Repository Evidence (scheduled_entry_ledger.json & cup_2027.json)
    entries = json.loads((EVIDENCE_DIR / "scheduled_entry_ledger.json").read_text(encoding="utf-8"))
    cup = json.loads((EVIDENCE_DIR / "cup_2027.json").read_text(encoding="utf-8"))

    completed_count = sum(1 for e in entries if e["completed"])
    no_telemetry = [e for e in entries if not e["has_telemetry"]]

    print("\n[PART B — ACTUAL REPOSITORY EVIDENCE: 227-ENTRY LEDGER & CUP_2027.JSON]")
    print(f"  Total scheduled boat-races in ledger : {len(entries)} (Completed: {completed_count}, Rate: {completed_count/len(entries)*100:.2f}%)")
    print(f"  Scheduled entries with 0 telemetry   : {len(no_telemetry)} -> {[ (e['race_id'], e['entry_id']) for e in no_telemetry ]}")
    assert len(entries) == 227 and completed_count == 169 and len(no_telemetry) == 3

    base_id = cup["scenario_analysis"]["baseline_scenario_id"]
    base_rec = next(r for r in cup["scenario_analysis"]["records"] if r["scenario_id"] == base_id)
    n_trials = base_rec["trials"]
    unresolved = base_rec["unresolved_trials"]
    resolved = base_rec["resolved_trials"]
    u_frac = unresolved / n_trials

    print(f"\n  AC38 Baseline Scenario ({base_id}):")
    print(f"    Trials = {n_trials}, Resolved = {resolved} ({resolved/n_trials*100:.2f}%), Unresolved = {unresolved} ({u_frac*100:.2f}%)")
    print(f"    Headline point probabilities: {cup['probabilities']}")
    assert all(v is None for v in cup["probabilities"].values())

    lower_sum = 0.0
    upper_sum = 0.0
    for team_id, bounds in cup["rule_bounds_by_team"].items():
        width = bounds["upper"] - bounds["lower"]
        lower_sum += bounds["lower"]
        upper_sum += bounds["upper"]
        print(
            f"    {team_id:14s} | Baseline Bounds: [{bounds['lower']*100:6.2f}%, {bounds['upper']*100:6.2f}%] "
            f"(Width = {width*100:.2f} pp) | 9-Scenario Envelope: "
            f"[{cup['scenario_envelope_by_team'][team_id]['lower']*100:5.2f}%, "
            f"{cup['scenario_envelope_by_team'][team_id]['upper']*100:5.2f}%]"
        )
        assert abs(width - u_frac) < 1e-12

    print(f"    Sum of lower bounds = {lower_sum:.4f} (equals 1 - u = {1.0 - u_frac:.4f})")
    print(f"    Sum of upper bounds = {upper_sum:.4f} (equals 1 + 6u = {1.0 + 6.0 * u_frac:.4f})")
    assert abs(lower_sum - (1.0 - u_frac)) < 1e-12
    assert abs(upper_sum - (1.0 + 6.0 * u_frac)) < 1e-12
    print("\nAll assertions passed successfully.")


if __name__ == "__main__":
    main()


EXECUTABLE EXAMPLE 03: PLACKETT-LUCE, TIED-N SUM & AC38 CUP RULE BOUNDS

[PART A.1 — SYNTHETIC 3-BOAT TIED-N EXPECTED RANK SUM IDENTITY]
  Boat 1 (gamma=3.0, p_comp=0.90) -> E[R_1] = 1.674750
  Boat 2 (gamma=2.0, p_comp=0.85) -> E[R_2] = 1.971500
  Boat 3 (gamma=1.0, p_comp=0.75) -> E[R_3] = 2.431250
  Standard permutation sum K(K+1)/2 : 6.000000
  Theoretical tied-N excess         : +0.077500
  Sum of expected ranks             : 6.077500

[PART A.2 — PERSISTENT VS INDEPENDENT SERIES WIN PROBABILITY (FIRST-TO-7)]
  50/50 mix of latent p in {0.50, 0.90} — Persistent ability draw : 75.00%
  50/50 mix of latent p in {0.50, 0.90} — Independent draws (p=0.7): 93.76%

[PART B — ACTUAL REPOSITORY EVIDENCE: 227-ENTRY LEDGER & CUP_2027.JSON]
  Total scheduled boat-races in ledger : 227 (Completed: 169, Rate: 74.45%)
  Scheduled entries with 0 telemetry   : 3 -> [('26_09_22_nap_ipr_pr2', 'GBR B'), ('26_09_22_nap_ipr_pr3', 'GBR B'), ('26_09_22_nap_ipr_pr4', 'GBR B')]

  AC38 Baseline Scenario (r